<a href="https://colab.research.google.com/github/BraedynL0530/autocaptcha/blob/master/SuperCoolCaptchaBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y tesseract-ocr
!pip install ultralytics fastapi uvicorn python-multipart pytesseract nest_asyncio

import asyncio
import time
from fastapi import FastAPI, UploadFile, File
from ultralytics import  YOLO
import cv2
import numpy as np
import pytesseract
import nest_asyncio
import uvicorn
import threading

app = FastAPI()

model = YOLO('yolo11n.pt')


@app.get("/")
async def root():
    return {"status": "alive"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    contents = await file.read()
    nparr = np.frombuffer(contents, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    if img is None:
        return {"error": "Failed to decode image"}

    results = model(img, iou=0.25) #that was easy

    detected_boxes = []
    for box in results[0].boxes:
        cx, cy, w, h = box.xywh.cpu().numpy()[0]
        classId = int(box.cls.cpu().tolist()[0])
        label = model.names[classId]
        detected_boxes.append({
            "cords": [cx,cy,w,h],
            "label": label
        })

    header = img[2:115, 3:386] #lol i forgot this isnt the full screen anymore
    gray_header = cv2.cvtColor(header, cv2.COLOR_BGR2GRAY)
    large_Header = cv2.resize(gray_header, None, fx=3, fy=3)
    inverted_header = cv2.bitwise_not(large_header)
    header = inverted_header
    config = r'--psm 11'
    captcha_prompt = pytesseract.image_to_string(header, lang='eng',config=config).lower()
    cv2.imwrite("ocr_input.png", header)
    print("prompt:",pytesseract.get_tesseract_version())


    return{
        "prompt":captcha_prompt,
        "boxes": detected_boxes
    }


def start_server():
    try:
         uvicorn.run(
            app,
            host="0.0.0.0",
            port=8000,
            log_level="debug"
        )
    except Exception as e:
        print(f"crashed: {e}")
thread = threading.Thread(
    target=start_server,
    daemon=True
)
print(threading.enumerate())
thread.start()

# test
time.sleep(5)
!curl http://localhost:8000

#run/ get link
!ssh -o StrictHostKeyChecking=no -R 80:localhost:8000 serveo.net

